# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. We will leverage the Croissant schema to load the dataset, inspect the schema using `@id` references, and perform basic exploratory data analysis with code examples you can extend further.

### Dataset Source
This notebook uses the Croissant schema at:

    https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Retrieve metadata as a native object
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Citation: {getattr(metadata, 'citeAs', 'N/A')}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section helps identify how to reference record sets and fields by their `@id`. The `mlcroissant` Dataset's metadata offers access to record sets and their schema definitions.

In [ ]:
# List out available record sets and their fields using `@id`.

record_sets = [rs for rs in getattr(metadata, 'recordSet', [])]
print(f"Number of record sets: {len(record_sets)}\n")
for idx, record_set in enumerate(record_sets):
    print(f"Record Set {idx+1}: \n    @id: {record_set['@id']}")
    rs_obj = dataset.metadata.find_by_id(record_set['@id'])
    # Fields extraction: use 'field' or 'fields' key based on metadata conventions
    fields = getattr(rs_obj, 'field', []) if hasattr(rs_obj, 'field') else []
    if not fields and hasattr(rs_obj, 'fields'): fields = getattr(rs_obj, 'fields')
    print(f"    Fields (@id):", end=" ")
    print([f['@id'] for f in fields])
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will demonstrate loading the data for each record set (if more are present), but will display the main protein quantification table as an example.

In [ ]:
# Collect available record set @ids for loading data
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
# Create a mapping from @id to DataFrame for easy referencing
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        print(f"    Columns: {df.columns.tolist()}")
        print(f"    Number of records: {len(df)}\n")
    else:
        df = pd.DataFrame()
        print(f"    [No records loaded for this record set]\n")
    dataframes[rs_id] = df

# Choose the primary table (first non-empty record set)
main_table_id = None
for rs_id in record_set_ids:
    if not dataframes[rs_id].empty:
        main_table_id = rs_id
        break

if main_table_id:
    print(f"Using record set '@id': {main_table_id} as main analysis table.\n")
    main_df = dataframes[main_table_id]
    display_cols = main_df.columns.tolist()
    print(f"Example columns: {display_cols}")
    display(main_df.head())
else:
    print("No data records found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

*Note:* Replace the following `@id`s for fields with those relevant to your selected table. The below code block demonstrates filtering, normalization, and grouping using `@id` as column keys.

In [ ]:
# EDA assumes a main table and at least one numeric column. Adjust @ids as relevant!
import numpy as np
if main_table_id and not main_df.empty:
    # Example: use the first numeric field found
    numeric_field_id = None
    for col in main_df.columns:
        # This is a heuristic: try to detect numeric fields
        if np.issubdtype(main_df[col].dropna().astype(str).str.replace(',','').str.replace('−', '-').str.replace('–', '-').str.replace('+',''), np.number):
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("No numeric field found for EDA.")
    else:
        print(f"Using '{numeric_field_id}' as the numeric field.\n")
        # Convert field to numeric (handle commas and missing values)
        main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id].astype(str).str.replace(',','').str.replace('−', '-').str.replace('–', '-').str.replace('+',''), errors='coerce')

        # Filter for records where the numeric field > a threshold
        threshold = main_df[numeric_field_id].quantile(0.90)  # Example: top 10%
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (90th percentile): {filtered_df.shape[0]} entries")

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field (use first column not numeric)
        group_field_id = None
        for col in main_df.columns:
            if col != numeric_field_id:
                if main_df[col].dtype == object and main_df[col].nunique() < main_df.shape[0]//2:
                    group_field_id = col
                    break

        if group_field_id:
            print(f"\nGrouping by '{group_field_id}'.\n")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print('\nNo suitable categorical field found for grouping.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
if main_table_id and not main_df.empty and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in main record set")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If group_field_id exists, plot group means
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        plot_df = main_df[[group_field_id, numeric_field_id]].dropna()
        medians = plot_df.groupby(group_field_id)[numeric_field_id].median().sort_values(ascending=False)
        sns.barplot(y=medians.index, x=medians.values, orient='h', palette='mako')
        plt.title(f"Median {numeric_field_id} by {group_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field_id)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to explore a complex, schema-driven FAIR dataset for mass spectrometry proteomics analysis. Using just the Croissant schema's URL, we can:
- Explore available record sets and fields by `@id`
- Load and examine tabular data
- Filter and normalize records using well-defined schema references
- Visualize quantitative features

This approach enables reproducible, transparent data loading and analysis directly from FAIR-compliant metadata. Explore deeper analyses by referencing fields using their `@id` as shown above.
